In [1]:
import numpy as np
import pandapower as pp
import pandapower.networks as pn
from energnn.graph import Graph

from pandapower_to_energnn.ready_to_use import (
    ACLoadFlowInputConverter,
    ACLoadFlowOutputConverter,
)


In [2]:
def _check_graph(converter, graph):
    print(graph)
    assert isinstance(graph, Graph)
    assert set(graph.hyper_edge_sets.keys()) == set(converter.elements_converter_dict.keys())

    for k, hyper_edge_set in graph.hyper_edge_sets.items():
        elements_converter = converter.elements_converter_dict[k]

        if elements_converter.feature_list is None:
            assert hyper_edge_set.feature_array is None
        else:
            assert set(hyper_edge_set.feature_names.keys()) == set(elements_converter.feature_list)
            feature_array = np.asarray(hyper_edge_set.feature_array)
            assert feature_array.shape[-1] == len(elements_converter.feature_list)
            assert np.all(np.isfinite(feature_array))

        if elements_converter.port_list is None:
            assert hyper_edge_set.port_dict is None
        else:
            assert set(hyper_edge_set.port_dict.keys()) == set(elements_converter.port_list)


def test_ac_load_flow_input(net):
    converter = ACLoadFlowInputConverter()
    _check_graph(converter, converter(net=net))


def test_ac_load_flow_output(net):
    converter = ACLoadFlowOutputConverter()
    _check_graph(converter, converter(net=net))


In [3]:
net = pn.case14()
pp.runpp(net)

In [4]:
test_ac_load_flow_input(net)

buses
                   ports
          energnn_adress
object_id               
0                      0
1                      1
2                      2
3                      3
4                      4
5                      5
6                      6
7                      7
8                      8
9                      9
10                    10
11                    11
12                    12
13                    13
ext_grids
          ports   features                
            bus in_service va_degree vm_pu
object_id                                 
0             0        1.0       0.0  1.06
gens
          ports   features             
            bus in_service  p_mw  vm_pu
object_id                              
0             1        1.0  40.0  1.045
1             2        1.0   0.0  1.010
2             5        1.0   0.0  1.070
3             7        1.0   0.0  1.090
lines
             ports           features                                    
          from_bus to_

In [5]:
test_ac_load_flow_output(net)

buses
           features
              vm_pu
object_id          
0          1.060000
1          1.045000
2          1.010000
3          1.017671
4          1.019514
5          1.070000
6          1.061520
7          1.090000
8          1.055932
9          1.050985
10         1.056906
11         1.055189
12         1.050382
13         1.035530
ext_grids
             features           
                 p_mw     q_mvar
object_id                       
0          232.393265 -16.549301
gens
            features           
              q_mvar  va_degree
object_id                      
0          43.557098  -4.982589
1          25.075348 -12.725100
2          12.730945 -14.220946
3          17.623451 -13.359628
lines
            features                                                          
           i_from_ka    i_to_ka   p_from_mw     p_to_mw q_from_mvar  q_to_mvar
object_id                                                                     
0           0.638290   0.634645  156.882

In [ ]:
# TODO: test with line bus_id's as floats
# TODO: drive development by pandapower-to-EnerGNN use cases
# TODO: EnerGNN to pandapower converter?